In [20]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])


0

In [21]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "new_models"
DATA_DIR    = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\user\OneDrive\Desktop\Praktikum\Final-Project\new_equity_forecasting_project


In [22]:
ref = pd.read_csv(RESULTS_DIR / "all_predictions.csv")
ref["Date"] = pd.to_datetime(ref["Date"], errors="coerce")
ref["cutoff_date"] = pd.to_datetime(ref["cutoff_date"], errors="coerce")
cutoffs = sorted(ref["cutoff_date"].dropna().dt.normalize().unique())
tickers = sorted(ref["Ticker"].unique())
print(f"Cutoffs : {len(cutoffs)}")
print(f"Tickers : {len(tickers)}")
print(f"Range   : {cutoffs[0].date()} -> {cutoffs[-1].date()}")

Cutoffs : 18
Tickers : 160
Range   : 2024-11-20 -> 2026-04-03


In [23]:
price_data = {}
for ticker in tickers:
    path = DATA_DIR / f"{ticker}.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path, parse_dates=["Date"], index_col="Date")
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    close_col = next((c for c in df.columns if "close" in c and "adj" not in c), None)
    if close_col is None:
        continue
    df = df.rename(columns={close_col: "close"})
    for col in ["open", "high", "low"]:
        match = next((c for c in df.columns if col in c), None)
        if match and match != col:
            df = df.rename(columns={match: col})
    df = df.sort_index().dropna(subset=["close"])
    price_data[ticker] = df

print(f"Loaded {len(price_data)} tickers")

Loaded 49 tickers


In [24]:
def make_features(df, date):
    hist = df[df.index <= pd.Timestamp(date)].copy()
    if len(hist) < 80:
        return None
    cl = hist["close"]

    ret_1m = float(cl.pct_change(21).iloc[-1])
    ret_3m = float(cl.pct_change(63).iloc[-1])
    ret_6m = float(cl.pct_change(126).iloc[-1]) if len(cl) >= 126 else ret_3m
    vol    = float(cl.pct_change().iloc[-21:].std() * np.sqrt(252))

    d    = cl.diff()
    gain = d.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-d).clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    rsi  = float((100 - 100 / (1 + gain / loss.replace(0, np.nan))).iloc[-1])

    cur   = float(cl.iloc[-1])
    ema20 = float(cl.ewm(span=20, adjust=False).mean().iloc[-1])
    ema50 = float(cl.ewm(span=50, adjust=False).mean().iloc[-1])
    price_vs_ema20 = (cur - ema20) / ema20
    ema_ratio      = ema20 / ema50 - 1

    if "high" in hist.columns and "low" in hist.columns:
        tr  = pd.concat([hist["high"] - hist["low"],
                         (hist["high"] - cl.shift()).abs(),
                         (hist["low"]  - cl.shift()).abs()], axis=1).max(axis=1)
        atr_norm = float(tr.ewm(alpha=1/14, adjust=False).mean().iloc[-1] / cur)
    else:
        atr_norm = vol / np.sqrt(252)

    feats = [ret_1m, ret_3m, ret_6m, vol, rsi, price_vs_ema20, ema_ratio, atr_norm]
    if any(not np.isfinite(v) for v in feats):
        return None
    return feats

In [25]:
def barrier_label(df, date, h_months, upper_pct, lower_pct):
    t     = pd.Timestamp(date)
    hist  = df[df.index <= t]["close"]
    if hist.empty:
        return np.nan
    start = float(hist.iloc[-1])
    fwd   = df[df.index > t]["close"].head(h_months * 22)
    if len(fwd) < 3:
        return np.nan
    for price in fwd:
        ret = price / start - 1
        if ret >=  upper_pct: return  1
        if ret <= -lower_pct: return -1
    final = float(fwd.iloc[-1]) / start - 1
    return 1 if final > 0 else (-1 if final < 0 else 0)

In [26]:
UPPER_H1 = 0.025
LOWER_H1 = 0.025
UPPER_H5 = 0.06
LOWER_H5 = 0.06

all_rows = []

for h_months, h_label, upper, lower in [
    (1, 1, UPPER_H1, LOWER_H1),
    (5, 5, UPPER_H5, LOWER_H5),
]:
    print(f"\n=== Horizon h={h_label} | barriers +/-{upper:.1%} ===")

    for i, cutoff in enumerate(cutoffs):
        train_end = cutoff - pd.DateOffset(months=2)
        X_train, y_train = [], []

        for ticker, df in price_data.items():
            hist = df[df.index <= train_end]
            if len(hist) < 100:
                continue
            sample_dates = hist.resample("MS").last().index[-30:]
            for sd in sample_dates:
                feats = make_features(df, sd)
                if feats is None:
                    continue
                lbl = barrier_label(df, sd, h_months, upper, lower)
                if np.isnan(lbl):
                    continue
                X_train.append(feats)
                y_train.append(int(lbl))

        if len(X_train) < 30:
            print(f"  [{i+1}/{len(cutoffs)}] {cutoff.date()} skipped ({len(X_train)} samples)")
            continue

        # --- Random Forest ---
        rf = RandomForestClassifier(n_estimators=150, max_depth=6,
                                    min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        rf_classes = list(rf.classes_)

        # --- XGBoost ---
        y_arr = np.array(y_train)
        label_map = {-1: 0, 0: 1, 1: 2}
        inv_map   = {0: -1, 1: 0, 2: 1}
        y_mapped  = np.array([label_map[y] for y in y_arr])

        xgb = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, eval_metric="mlogloss",
                             verbosity=0, n_jobs=-1)
        xgb.fit(X_train, y_mapped)

        n_pred = 0
        for ticker, df in price_data.items():
            feats = make_features(df, cutoff)
            if feats is None:
                continue

            ref_row = ref[
                (ref["Ticker"] == ticker) &
                (ref["cutoff_date"].dt.normalize() == cutoff) &
                (ref["Horizon"] == h_label)
            ]
            if ref_row.empty:
                continue

            base = {
                "Date":         ref_row["Date"].iloc[0],
                "Ticker":       ticker,
                "Horizon":      h_label,
                "y_true":       float(ref_row["y_true"].iloc[0]),
                "cutoff_date":  cutoff.strftime("%Y-%m-%d"),
                "last_close":   np.nan,
                "pred_close":   np.nan,
                "actual_close": np.nan,
            }

            # RF signal
            rf_proba  = rf.predict_proba([feats])[0]
            p_up_rf   = rf_proba[rf_classes.index( 1)] if  1 in rf_classes else 0.0
            p_dn_rf   = rf_proba[rf_classes.index(-1)] if -1 in rf_classes else 0.0
            all_rows.append({**base, "Model": "BarrierRF",  "y_pred": float(p_up_rf - p_dn_rf)})

            # XGB signal
            xgb_proba = xgb.predict_proba([feats])[0]
            p_up_xgb  = xgb_proba[2]   # class 2 → original label +1
            p_dn_xgb  = xgb_proba[0]   # class 0 → original label -1
            all_rows.append({**base, "Model": "BarrierXGB", "y_pred": float(p_up_xgb - p_dn_xgb)})

            n_pred += 1

        print(f"  [{i+1}/{len(cutoffs)}] {cutoff.date()} — {len(X_train)} train samples, {n_pred} predictions")

print(f"\nTotal rows: {len(all_rows)}  (BarrierRF + BarrierXGB)")


=== Horizon h=1 | barriers +/-2.5% ===


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got [0 2]

In [ ]:
barrier_df = pd.DataFrame(all_rows)
out = RESULTS_DIR / "barrier_predictions.csv"
barrier_df.to_csv(out, index=False)
print(f"Saved  : {out}")
print(f"Shape  : {barrier_df.shape}")
print(f"\nModels in file:")
print(barrier_df["Model"].value_counts())
print(barrier_df.groupby("Model")[["y_true","y_pred"]].describe().round(4))

In [ ]:
from scipy.stats import spearmanr
print("=== Quick RankIC check ===")
for model_name in ["BarrierRF", "BarrierXGB"]:
    sub = barrier_df[barrier_df["Model"] == model_name]
    print(f"\n  {model_name}:")
    for h in [1, 5]:
        hsub = sub[sub["Horizon"] == h]
        rics = []
        for cutoff, g in hsub.groupby("cutoff_date"):
            g = g.dropna(subset=["y_true", "y_pred"])
            if len(g) >= 5:
                r, _ = spearmanr(g["y_pred"], g["y_true"])
                if np.isfinite(r):
                    rics.append(r)
        if rics:
            print(f"    h={h}: mean RankIC = {np.mean(rics):.4f}  std = {np.std(rics):.4f}  ({len(rics)} cutoffs)")
        else:
            print(f"    h={h}: no valid cutoffs")
print("\nRe-run notebook 04_final_analysis.ipynb to add both models to all charts.")